In [ ]:
#| default_exp measurement

In [ ]:
#| export
from __future__ import annotations

import json
import math
import numbers
import re
from datetime import datetime
from pathlib import Path

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

A measurement row is one session of one model variant on one device and software stack, at one precision
and one batch size, on one date. Rows are stored as JSON Lines, one object per line, appended and never
rewritten; `to_parquet` writes a set of rows as one table for publishing. `FIELDS` is the schema: each
field's type, whether it is required, and the rules its value obeys. A field that is not required may be
absent or `None`, which means *not recorded*.

| Group | Fields |
|---|---|
| identity and provenance | `schema_version`, `run_id`, `measured_at` (ISO-8601, UTC), `series`, `harness_commit`, `fasterai_version`, `report_url`, `hf_repo` |
| model | `model_id`, `model_revision`, `task`, `architecture`, `base_weights`, `weights_license`, `variant`, `recipe`, `input_resolution`, `macs`, `params`, `model_hash`, `engine_hash`, `size_mb` |
| device and runtime | `device`, `device_host`, `runtime`, `runtime_version`, `platform_version`, `precision` (what ran), `int8_mode` (iff `int8`), `power_mode`, `clocks_locked`, `batch_size` |
| conditions | `temp_start_c`, `temp_end_c`, `gpu_freq_mhz`, `throttled`, `warmup_s`, `duration_s`, `repeats`, `latency_spread_pct` |
| latency | `latency_ms_p50`, `latency_ms_p90`, `latency_ms_p99`, `latency_scope`, `host_latency_ms`, `latency_source` |
| quality | `metric`, `eval_set`, `eval_set_hash`, `dataset_license`, `eval_n`, `accuracy`, `ci_low`, `ci_high`, `ci_method`, `measured_through`, `argmax_parity`, `parity_floor`, `baseline_run_id` |

Throughput is derived, never stored: `throughput(row)` is `batch_size * 1000 / latency_ms_p50`.

- `series` is a free-text label grouping the rows measured in one session, typically one night; it is not part of the key.
- `latency_scope` says what the latency covers, e.g. engine execution only, or host transfers included; two rows are only comparable when it matches.
- `engine_hash` is pinned to the sha256 of the engine file that ran; `model_hash` is whatever fingerprint the exporter records, so it is only a string.

Two questions are kept apart: `validate` asks whether a row is **well formed**, `status` whether the
measurement it records is **valid**. Neither compares a row to a latency or accuracy budget: a row carries no
verdict column, and whoever reads it recomputes their own. A well-formed row is `INVALID` when:

- `measured_through` is not `deployed_engine`: the accuracy was not measured through the engine that was timed;
- `latency_source` is not `measured`: the latency was not measured on the device;
- `argmax_parity` is below `parity_floor`;
- `throttled` is true;
- `latency_spread_pct` is above `max_spread_pct` (5 by default);
- any of the last three is not recorded.

In [ ]:
#| export
SCHEMA_VERSION = 1
_UTC = r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(\.\d+)?(Z|\+00:00)'
_PCT = dict(ge=0, le=100)

# name -> type, required, and the declarative rules its value obeys; a field that is not required may be absent or None
FIELDS = {
    # identity and provenance
    'schema_version': dict(type=int, required=True, choices=(SCHEMA_VERSION,)),
    'run_id': dict(type=str, required=True),
    'measured_at': dict(type=str, required=True, pattern=_UTC, date=True),
    'series': dict(type=str, required=False),
    'harness_commit': dict(type=str, required=True),
    'fasterai_version': dict(type=str, required=False),
    'report_url': dict(type=str, required=False),
    'hf_repo': dict(type=str, required=False),
    # model
    'model_id': dict(type=str, required=True),
    'model_revision': dict(type=str, required=True),
    'task': dict(type=str, required=True, choices=('classification', 'detection')),
    'architecture': dict(type=str, required=True),
    'base_weights': dict(type=str, required=True),
    'weights_license': dict(type=str, required=True),
    'variant': dict(type=str, required=True),
    'recipe': dict(type=dict, required=True, json=True),
    'input_resolution': dict(type=int, required=True, gt=0),
    'macs': dict(type=int, required=True, gt=0),
    'params': dict(type=int, required=True, gt=0),
    'model_hash': dict(type=str, required=False),   # whatever fingerprint the exporter records, so not pinned
    'engine_hash': dict(type=str, required=True, pattern=r'[0-9a-f]{64}'),   # sha256 of the engine file that ran
    'size_mb': dict(type=float, required=True, gt=0),
    # device and runtime
    'device': dict(type=str, required=True),
    'device_host': dict(type=str, required=True, choices=('own', 'rented', 'partner')),
    'runtime': dict(type=str, required=True, choices=('tensorrt', 'openvino', 'onnxruntime', 'hailo', 'axelera')),
    'runtime_version': dict(type=str, required=True),
    'platform_version': dict(type=str, required=False),
    'precision': dict(type=str, required=True, choices=('fp32', 'fp16', 'int8', 'mixed')),
    'int8_mode': dict(type=str, required=False, choices=('implicit', 'explicit')),
    'power_mode': dict(type=str, required=False),
    'clocks_locked': dict(type=bool, required=False),
    'batch_size': dict(type=int, required=True, ge=1),
    # conditions
    'temp_start_c': dict(type=float, required=False),
    'temp_end_c': dict(type=float, required=False),
    'gpu_freq_mhz': dict(type=float, required=False, gt=0),
    'throttled': dict(type=bool, required=False),
    'warmup_s': dict(type=float, required=False, ge=0),
    'duration_s': dict(type=float, required=False, ge=0),
    'repeats': dict(type=int, required=True, ge=1),
    'latency_spread_pct': dict(type=float, required=False, ge=0),
    # latency
    'latency_ms_p50': dict(type=float, required=True, gt=0),
    'latency_ms_p90': dict(type=float, required=False, gt=0),
    'latency_ms_p99': dict(type=float, required=False, gt=0),
    'latency_scope': dict(type=str, required=True),
    'host_latency_ms': dict(type=float, required=False, gt=0),
    'latency_source': dict(type=str, required=True, choices=('measured', 'estimated')),
    # quality
    'metric': dict(type=str, required=True),
    'eval_set': dict(type=str, required=True),
    'eval_set_hash': dict(type=str, required=True),
    'dataset_license': dict(type=str, required=True),
    'eval_n': dict(type=int, required=True, gt=0),
    'accuracy': dict(type=float, required=True, **_PCT),
    'ci_low': dict(type=float, required=True, **_PCT),
    'ci_high': dict(type=float, required=True, **_PCT),
    'ci_method': dict(type=str, required=True, choices=('wilson', 'bootstrap')),
    'measured_through': dict(type=str, required=True, choices=('deployed_engine', 'dev_model')),
    'argmax_parity': dict(type=float, required=False, **_PCT),
    'parity_floor': dict(type=float, required=True, **_PCT),
    'baseline_run_id': dict(type=str, required=False),
}

def _item(o):
    "A numpy scalar as the Python number JSON knows"
    if isinstance(o, numbers.Number) and hasattr(o, 'item'): return o.item()
    raise TypeError(f'{type(o).__name__} is not JSON serializable')


def _json(obj, **kw): return json.dumps(obj, default=_item, allow_nan=False, **kw)
def _canonical(d): return _json(d, sort_keys=True, separators=(',', ':'))


def _is_json(v):
    try: return _json(v) is not None
    except (TypeError, ValueError): return False


def _is_date(v):
    try: return datetime.fromisoformat(v.replace('Z', '+00:00')) is not None   # Python 3.10 does not read 'Z'
    except ValueError: return False


_RULES = {
    'choices': (lambda v, c: v in c, 'not one of {}'),
    'gt': (lambda v, b: v > b, 'not > {}'),
    'ge': (lambda v, b: v >= b, 'not >= {}'),
    'le': (lambda v, b: v <= b, 'not <= {}'),
    'pattern': (lambda v, p: re.fullmatch(p, v) is not None, 'does not match {}'),
    'date': (lambda v, _: _is_date(v), 'is not a date'),
    'json': (lambda v, _: _is_json(v), 'is not JSON'),
}
_ORDERED = (('ci_low', 'accuracy', 'ci_high'), ('latency_ms_p50', 'latency_ms_p90', 'latency_ms_p99'))


def _typed(v, t):
    "`v` is a `t`; a bool is not a number, a numpy integer is an int, an int is a float, a float is finite"
    if isinstance(v, bool): return t is bool
    if t is int: return isinstance(v, numbers.Integral)
    if t is float: return isinstance(v, numbers.Real) and math.isfinite(v)
    return isinstance(v, t)


def _field_problems(name, v):
    "The problems of one field value, against its spec"
    spec = FIELDS[name]
    if v is None: return [f'{name}: missing'] if spec['required'] else []
    if not _typed(v, spec['type']): return [f'{name}: expected {spec["type"].__name__}, got {v!r}']
    return [f'{name}: {v!r} {msg.format(spec[r])}' for r, (ok, msg) in _RULES.items() if r in spec and not ok(v, spec[r])]


def _cross_problems(row):
    "Problems between fields; each field involved is already well typed"
    p, mode = row['precision'], row.get('int8_mode')
    out = []
    if p == 'int8' and mode is None: out.append('int8_mode: missing, required when precision is int8')
    if p != 'int8' and mode is not None: out.append(f'int8_mode: {mode!r} given for precision {p!r}')
    for fields in _ORDERED:
        vals = [(f, row[f]) for f in fields if row.get(f) is not None]
        if [v for _, v in vals] != sorted(v for _, v in vals):
            out.append(f"{', '.join(fields)}: not in order, " + ', '.join(f'{f}={v}' for f, v in vals))
    return out


def validate(
    row: dict,  # one measurement row
) -> list[str]:
    "Every structural problem of `row`, each prefixed by its field name; an empty list means well formed"
    if not isinstance(row, dict): return [f'row: expected a dict, got {type(row).__name__}']
    out = [f'{k}: unknown field' for k in row if k not in FIELDS]
    for name in FIELDS: out += _field_problems(name, row.get(name))
    return out or _cross_problems(row)

In [ ]:
show_doc(validate)

In [ ]:
#| export
# field, when its value makes the measurement invalid, and why; a field that is None is not recorded
_VALIDITY = (
    ('measured_through', lambda v, r, m: v != 'deployed_engine', 'accuracy not measured through the deployed engine'),
    ('latency_source', lambda v, r, m: v != 'measured', 'latency not measured on the device'),
    ('argmax_parity', lambda v, r, m: v < r['parity_floor'], '{v} below parity_floor {r[parity_floor]}'),
    ('throttled', lambda v, r, m: v, 'the device throttled during the session'),
    ('latency_spread_pct', lambda v, r, m: v > m, '{v} above {m}'),
)


def status(
    row: dict,                    # one measurement row
    max_spread_pct: float = 5.0,  # largest (max-min)/median latency spread over repeats, in percent
) -> tuple[str, list[str]]:
    "('VALID', []) if `row` is a usable measurement, else ('INVALID', reasons); not a verdict against any budget"
    reasons = validate(row)
    if reasons: return 'INVALID', reasons
    for f, bad, why in _VALIDITY:
        v = row.get(f)
        if v is None: reasons.append(f'{f}: not recorded')
        elif bad(v, row, max_spread_pct): reasons.append(f'{f}: ' + why.format(v=v, r=row, m=max_spread_pct))
    return ('INVALID', reasons) if reasons else ('VALID', [])

In [ ]:
show_doc(status)

An unrecorded condition is not a clean one: a row that does not say whether the device throttled, what
its latency spread was, or how its outputs agreed with the reference is `INVALID`, with the reason
`not recorded`.

In [ ]:
#| export
def key(
    row: dict,  # one measurement row
) -> tuple:
    "The identity a row is unique on; a software update on the device gives a new key"
    return (row['model_id'], row['model_revision'], row['variant'], _canonical(row['recipe']),
            row['input_resolution'], row['device'], row['runtime'], row['runtime_version'],
            row.get('platform_version'), row['precision'], row.get('int8_mode'), row['batch_size'],
            row['measured_at'])


def throughput(
    row: dict,  # one measurement row
) -> float:
    "Images per second at the timed batch, derived from the median latency and never stored"
    return row['batch_size'] * 1000 / row['latency_ms_p50']

In [ ]:
show_doc(key)

In [ ]:
show_doc(throughput)

In [ ]:
#| export
def _parse(text, path):
    "The rows of a JSON Lines text; a malformed line raises with its number"
    rows = []
    for i, line in enumerate(text.splitlines(), 1):
        if not line.strip(): continue
        try: r = json.loads(line)
        except json.JSONDecodeError as e: raise ValueError(f'{path}:{i}: not JSON ({e.msg})') from None
        if not isinstance(r, dict): raise ValueError(f'{path}:{i}: not a JSON object')
        rows.append(r)
    return rows


def read_rows(
    path: str | Path,  # a JSON Lines file of measurement rows
) -> list[dict]:
    "The rows of `path`, as stored; `status` judges them"
    return _parse(Path(path).read_text(), path)


def write_row(
    path: str | Path,  # a JSON Lines file, created if absent
    row: dict,         # the measurement row to append
):
    "Append `row` to `path`; refuses a malformed row and a row whose key is already in the file"
    problems = validate(row)
    if problems: raise ValueError('row refused: ' + '; '.join(problems))
    p, k = Path(path), key(row)
    text = p.read_text() if p.exists() else ''
    if k in {key(r) for r in _parse(text, path)}: raise ValueError(f'row refused: key {k} already in {path}')
    # ponytail: no file lock, one writer at a time
    with p.open('a') as f: f.write(('\n' if text and not text.endswith('\n') else '') + _json(row) + '\n')

In [ ]:
show_doc(read_rows)

In [ ]:
show_doc(write_row)

In [ ]:
#| export
def to_parquet(
    rows: list[dict],  # well-formed measurement rows, e.g. from `read_rows`
    path: str | Path,  # the Parquet file to write
):
    "Write `rows` as one Parquet table, one column per field in `FIELDS` order, dict fields as canonical JSON"
    try:
        import pandas as pd
        import pyarrow  # noqa: F401
    except ImportError: raise ImportError("to_parquet needs pandas and pyarrow: pip install 'fastermodels[parquet]'") from None
    for i, r in enumerate(rows):
        problems = validate(r)
        if problems: raise ValueError(f'row {i} refused: ' + '; '.join(problems))
    table = [{k: _canonical(r[k]) if FIELDS[k]['type'] is dict else r.get(k) for k in FIELDS} for r in rows]
    pd.DataFrame(table, columns=list(FIELDS)).to_parquet(path, engine='pyarrow', index=False)

In [ ]:
show_doc(to_parquet)

`to_parquet` needs the optional `parquet` extra: `pip install 'fastermodels[parquet]'`.

---

## Usage

```python
from fastermodels import measurement

row = dict(schema_version=1, run_id='2026-09-28-a', measured_at='2026-09-28T10:00:00Z',
           harness_commit='deadbee', model_id='org/resnet50-p30', model_revision='a1b2c3d',
           task='classification', architecture='resnet50', base_weights='timm/resnet50.a1_in1k',
           weights_license='apache-2.0', variant='p30-int8', recipe={'prune': 0.30, 'quant': 'int8', 'calib_n': 512},
           input_resolution=224, macs=2_900_000_000, params=17_000_000, engine_hash='0' * 64, size_mb=18.2,
           device='jetson-nano-4gb', device_host='own', runtime='tensorrt', runtime_version='8.2.1',
           platform_version='jetpack-4.6.1', precision='int8', int8_mode='explicit', batch_size=1,
           throttled=False, repeats=5, latency_spread_pct=1.2, latency_ms_p50=2.1,
           latency_scope='engine execute, host transfers excluded', latency_source='measured',
           metric='top1', eval_set='imagenet-val', eval_set_hash='sha256:...', dataset_license='custom',
           eval_n=50_000, accuracy=78.9, ci_low=78.5, ci_high=79.3, ci_method='wilson',
           measured_through='deployed_engine', argmax_parity=99.7, parity_floor=99.0)

measurement.validate(row)                 # [] : well formed
measurement.status(row)                   # ('VALID', [])
measurement.throughput(row)               # 476.19... images per second, derived
measurement.write_row('rows.jsonl', row)  # appends one line; a second call with the same row is refused
measurement.to_parquet(measurement.read_rows('rows.jsonl'), 'rows.parquet')
```

---

## See Also

- [Model](00_model.html) - the published model a row measures
- [Eval](01_eval.html) - `wilson`, the interval `ci_method='wilson'` refers to, and `agreement`
- [Card](02_card.html) - where a published latency is written down
- [Gate](03_gate.html) - the conditions an artifact meets before it is published

Tests live in `nbs/tests/test_measurement.ipynb`.